# Connaissance et pratique de la Science Ouverte

In [ ]:
import pandas as pd

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket
import scipy.stats

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")

list_affil = pd.read_csv("list_affiliation.csv", sep =",")
df0 = df0.loc[~df0.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2"])].merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")

In [ ]:

df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")


In [ ]:
list_nominal_simple = [x for x in df_col.label.loc[(df_col.type.isin(["simple_nominal", "booléen","ordinal"]))]]


In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [ ]:
def grouped_question(data, column, index = "q45_clé"):
    """


    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [ ]:
df0

In [ ]:
df_col["question_family"] = df_col.label.apply(lambda row : row.split("_")[0])

dict_question = dict(zip(df_col.label.loc[df_col.label.isin(list_nominal_simple)], df_col.question_family.loc[df_col.label.isin(list_nominal_simple)]))


Les premières questions interrogeaient les participants sur leurs connaissances et leurs pratiques de la Science Ouverte, thème central de l'enquête. Le questionnaire commencait ainsi par demander aux participants de préciser leur degré de familiarité avec les principes de la Science Ouverte. 56% ont répond "oui, un peu", 32 "oui, tout à fait" et 12% "non". Les pratiques étaient ensuite abordaient à travers des questions sur le dépôt des publications sur Hal et Octaviana, et l'ouverture de données de recherche.

40 % des personnes interrogées ont déposé plusieurs fois leurs publications sur Hal et 35% le font systématiquement. 13%  ne l'ont jamais fais. Concernant Octaviana, un peu moins de 10% des répondants ont déjà déposé des productions sur la bibliothèque numérique. Ces productions sont des captations d'événements pour trois d'entre elles et des travaux universitaires et des publications pour les huit autres. En tout une personne interrogée sur cinq n'a jamais entendu parlé d'Octaviana.

| Principes de la science ouverte   |   nb |    freq |
|:-------------------|-----:|--------:|
| Non                |   14 | 11,8 |
| Oui, tout à fait   |   38 | 31,9 |
| Oui, un peu        |   67 | 56,3 |
|Total               |119   | 100,0|


|Dépôt dans Hal        |   nb |    freq |
|:----------------------|-----:|--------:|
| Non                   |   15 | 12,6  |
| Oui, plusieurs fois   |   48 | 40,3 |
| Oui, systématiquement |   41 | 34,5 |
| Oui, une fois         |   15 | 12,6 |
|Total               |119   | 100,0|


| Dépot sur Octaviana  |   nb |    freq |
|:----------------|-----:|--------:|
| Non             |  108 | 90,8 |
| Oui             |   11 |  9,2|
|Total               |119   | 100,0|

| Ouverture des données   |   nb |   total |      freq |  
|:------------------------|-----:|--------:|----------:|
| Non                     |   86 |     119 | 72,3  |      
| Autre, précisez         |   24 |     119 | 20,7   |          
| Oui, sur Nakala         |    5 |     119 |  4,2  |          
| Oui, sur Zenodo |    4 |     119 |  3,4  |         
| Oui, sur Data Paris 8   |    1 |     119 |  0,8 |        
  


Enfin, 28% des personnes interrogées ont déjà produit des données ouvertes. 8% d'entre elles ont utilisé au moins une fois des entrepôts généralistes comme Nakala (N=5), Zenodo (N=4) ou Data Paris 8 (N=1). Les autres modalité d'ouverture des données sont Open Science Framework (N=10), des sites web personnels (N=6), des dépôts "git" (N=3) et des entrepots spécialisés (N=3) liés à la linguistiques : Ortolang et Cocoon.

![](viz/autre_entrepot_rec.png)



In [ ]:
!pip install tabulate

In [ ]:
so_principe = grouped_question(df0, column="q1_so_principles", index = "q45_clé")
print(so_principe[["q1_so_principles", "nb","freq"]].to_markdown(index=False))
depot_hal = grouped_question(df0, column="q2_hal_depot", index = "q45_clé")
print("\n### Dépôt dans hal\n", depot_hal[["q2_hal_depot", "nb","freq"]].to_markdown(index=False))
octaviana = grouped_question(df0, column="q3_octavi_rec", index = "q45_clé")
print("\n### Dépôt dans Octaviana\n", octaviana[["q3_octavi_rec", "nb","freq"]].to_markdown(index=False))

df_exp, octaviana2 = split_multiple_choices(df0, column="q3_octavi_depot", index="q45_clé", sep = '|')
print("\n### Dépôt dans Octaviana détaillé \n", octaviana2.to_markdown())



In [ ]:
df0.loc[df0.q4_diff_data !="Non", "q4_diff_data_grouped"] = "Oui"
df0.loc[df0.q4_diff_data =="Non", "q4_diff_data_grouped"] = "Non"

depot_donnee = grouped_question(df0, column="q4_diff_data_grouped", index = "q45_clé")
print("\n### Dépôt des données de recherche\n", depot_donnee)

df_exp, depot_donnee2 = split_multiple_choices(df0, column="q4_diff_data", index="q45_clé", sep = '|')
print("\n### Dépôt dans Octaviana détaillé \n", depot_donnee2.to_markdown(index=False))

## Les autres modes de dépôt des données de recherche

In [ ]:
df0.loc[df0["q4_diff_data"]=='Autre, précisez', "q4_diff_data_rec"] = df0.q4_autres_entrepots_rec
df0.loc[df0["q4_diff_data"]!='Autre, précisez', "q4_diff_data_rec"] = df0.q4_diff_data
df0.loc[df0["q4_diff_data_rec"].isna(), "q4_diff_data_rec"] = "NSP"
q4 = split_multiple_choices(df0, column="q4_diff_data_rec", index="q45_clé", sep = '|')

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(6,4))

# Plot the total crashes
sns.set_color_codes("pastel")
q4_other = df0.loc[df0.q4_diff_data=="Autre, précisez"].fillna("Non précisé")
col= "q4_autres_entrepots_rec"
df_exp, gb_data = split_multiple_choices(q4_other, column=col, index="q45_clé", sep = '|')
sns.barplot(x="total", y=col, data=gb_data,
            label="Non", color="b", ax=ax)
sns.barplot(x="nb", y=col, data=gb_data,
            label="Oui", color="r", ax=ax)
titre = "Les autres lieux de dépot des données de recherche"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/autre_entrepot_rec.png", bbox_inches='tight', dpi = 200)



## Les besoins d'accompagnement à la Science Ouverte

Mis à part l'accompagnement au dépot dans Hal considéré comme inutile par 68% des répondants, une majorité d'entre eux sont favorables aux accompagnement à la Science Ouverte comme étant utiles. Les licences de diffusion des résultats de recherche est le sujet pour lequel les répondants ont manifesté le plus d'intérêt : 87% considèrent qu'un accompagnement sur ce thème serait "plutôt utile" (52%) voire "très utile" (35%). Ensuite, 81% des participants à l'enquêtre déclarent qu'il serait utile d'être accompagner dans le processus permettant de produire des données respectant les principes FAIR (Facile à trouver, Accessible, Interopérable, et Réutilisable). De façon général, les accompagnements apparaissent d'autant plus utiles qu'ils portent sur des aspects juridiques (licence, RGPD) ou des sujets liés à l'ouverture des données (métadonnées, dépots, principes FAIR).

![](viz/util_accomp_so.png)


In [ ]:
for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]:
    gb_data= grouped_question(df0, column=col, index="q45_clé")
    print(gb_data.to_markdown())

In [ ]:
[col for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]]

In [ ]:
dict_accomp = {'q8_depot_hal_help':"Dépot sur Hal",
 'q8_rediger_pdg_help': 'Rédiger un PGD',
 'q8_respect_rgpd_help': 'Respecter le RGPD',
 'q8_licences_help': 'Choix des licences de diffusion',
 'q8_depot_donnees_help': 'Déposer des données',
 'q8_metadonnees_help': 'Connaître les standards de métadonnées',
 'q8_data_paper_help':'Rédiger un data paper',
 'q8_fair_data_help':'Fairiser les données'
              }

In [ ]:

i=-1
for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]:
    i+=1
    dftmp = df0[["q45_clé", col]].rename(columns={col:"interet_accomp"})
    dftmp["accomp_name"]= dict_accomp[col]


    if i == 0:
        q8 = dftmp.copy()
    else:
        q8 = pd.concat([q8, dftmp])


In [ ]:
q8_dis = pd.crosstab(q8.accomp_name, q8.interet_accomp, normalize="index").cumsum(axis=1).stack().reset_index(name='nb').sort_values(by=["accomp_name","interet_accomp"],
                                                                                                                                     ascending=[True, False])


In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

sns.set_theme(style="ticks", context="paper")

   
g = sns.barplot(x="nb", y="accomp_name", data=q8_dis, hue="interet_accomp", dodge=False)

titre = "L'utilité des accompagnements à la science ouverte"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_xticks(ticks = [x/10 for x in range(11)])
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
ax.legend(bbox_to_anchor=(1.05, 1),
                         loc='upper left', borderaxespad=0.)

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/util_accomp_so.png", bbox_inches='tight', dpi = 200)

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

sns.set_color_codes("pastel")

q8_grp = q8.groupby(["accomp_name","interet_accomp"]).agg(nb=("q45_clé", "size")).sort_values(by=["accomp_name","nb"], ascending=[True, False])

    
sns.barplot(x="nb", y="accomp_name", data=q8_grp, hue="interet_accomp", dodge=True)
titre = "L'utilité des accompagnements à la science ouverte"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
#plt.savefig(f"viz/autre_entrepot_rec.png", bbox_inches='tight', dpi = 200)